# Phase 7b — Model Experiments: other boosters, deep learning, and a time-saving cascade

**Question.** Phase 7 picked `HistGradientBoostingClassifier` (hold-out PR-AUC 0.9972, F0.5 0.9504). Is there a model that does **better**, and can we **save time** at test scale (~68M candidate pairs)?

**Experiments (all trained on the 80 % fit queries, scored on the 20 % hold-out — the same split as Phases 5–7; validation labels are not used for choosing).**
1. Reference: tuned HistGradientBoosting (Phase 7 parameters).
2. Other boosters: LightGBM, XGBoost, CatBoost.
3. **Deep learning, pointwise:** an MLP (3 hidden layers, **ReLU**, batch-norm, dropout, 2-class **softmax** output, cross-entropy loss, trained by **back-propagation** with Adam).
4. **Deep learning, set-aware ("DeepSets"):** the same MLP encodes every candidate; each candidate is then scored together with the **mean and max of the other candidates of its query** — a learned version of the cluster-consistency idea.
5. Blends of the best of the above.
6. **Cascade for speed:** a tiny first-stage model on the *free* retrieval features (scores, ranks) removes obvious non-matches **before** the expensive pair features are computed.

**Decision rule (fixed beforehand).** A challenger replaces HistGradientBoosting only if ΔPR-AUC ≥ +0.002 **and** Δ best-threshold macro F0.5 ≥ +0.003 **and** the paired-bootstrap 95 % interval of ΔF0.5 is above 0 — the same rule used in Phase 6.
Speed: a model is also reported with its prediction time per million pairs.
Fairness note: HistGradientBoosting had a random-search tuning in Phase 7; the other models get sensible settings and a small number of variants, so a close result is a tie, not a win.

**Sections:** 1 Load → 2 Boosters → 3 MLP (deep learning) → 4 DeepSets → 5 Blends + decision → 6 Cascade → 7 Summary

In [1]:
import os, sys, json, time, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
warnings.filterwarnings("ignore")
sys.path.insert(0, os.getcwd())
import eval_utils as EU
ROOT = "../../../"; INTERIM = ROOT + "dataset/interim"; SEED = 42
FL = json.load(open("feature_list.json")); FEATS = FL["final_features"]; MC = json.load(open("model_config.json"))
pd.set_option("display.width", 170)
CACHE = INTERIM + "/phase7b_hold_scores.pkl"; HS = pickle.load(open(CACHE, "rb")) if os.path.exists(CACHE) else {}
TIMES = {}
def save(): pickle.dump(HS, open(CACHE, "wb"))

## 1. Load train data and the Phase 7 hold-out split

In [2]:
ctr = pd.read_pickle(INTERIM + "/candidates_train.pkl"); Fall = pd.read_pickle(INTERIM + "/features_train.pkl")
P4 = pickle.load(open(INTERIM + "/phase4_truth_and_queries.pkl", "rb")); ntrue = P4["truth"]["train"].groupby("s1_id").size()
ents = P4["queries"][P4["queries"].split == "train"].s1_id.tolist()
y = ctr.label.values; qid = ctr.s1_id.values
rng = np.random.default_rng(SEED); hold = set(rng.choice(np.array(ents), int(0.2 * len(ents)), replace=False)); is_hold = ctr.s1_id.isin(hold).values
ent_hold = [e for e in ents if e in hold]
X = Fall[FEATS].values.astype(np.float32); Xf, yf, qf = X[~is_hold], y[~is_hold], qid[~is_hold]; Xh, yh, qh = X[is_hold], y[is_hold], qid[is_hold]
print(f"fit {len(yf):,} pairs / {len(set(qf)):,} queries | hold-out {len(yh):,} pairs / {len(ent_hold):,} queries | {len(FEATS)} features | positives {y.mean():.2%}")
def evaluate(name, p):
    t, f, _ = EU.best_threshold(ent_hold, ntrue, qh, yh, p, grid=np.round(np.arange(0.1, 0.96, 0.025), 3))
    return {"model": name, "PR-AUC": average_precision_score(yh, p), "best_thr": t, "macro_F0.5": f}
def fit_score(name, make, fit, predict):
    if name in HS: return
    t0 = time.time(); m = make(); fit(m); tf = time.time() - t0; t0 = time.time(); p = predict(m, Xh); tp = time.time() - t0
    HS[name] = np.asarray(p, np.float64); TIMES[name] = {"fit_s": round(tf), "predict_s_per_M": round(tp / len(Xh) * 1e6, 1)}; save(); print(f"{name}: fit {tf:.0f}s, predict {tp:.1f}s")

fit 728,784 pairs / 19,200 queries | hold-out 181,999 pairs / 4,800 queries | 69 features | positives 8.52%


## 2. Gradient-boosting family

In [3]:
gp = MC["gbdt"]["params"]
fit_score("hist_gbm (Phase 7)", lambda: HistGradientBoostingClassifier(random_state=SEED, **gp), lambda m: m.fit(Xf, yf), lambda m, X_: m.predict_proba(X_)[:, 1])
import lightgbm as lgb
fit_score("lightgbm", lambda: lgb.LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=127, min_child_samples=100, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=10, n_jobs=2, random_state=SEED, verbose=-1),
          lambda m: m.fit(Xf, yf), lambda m, X_: m.predict_proba(X_)[:, 1])
import xgboost as xgb
fit_score("xgboost", lambda: xgb.XGBClassifier(n_estimators=500, learning_rate=0.08, max_depth=8, subsample=0.8, colsample_bytree=0.8, min_child_weight=5, reg_lambda=10, tree_method="hist", n_jobs=2, random_state=SEED),
          lambda m: m.fit(Xf, yf), lambda m, X_: m.predict_proba(X_)[:, 1])
from catboost import CatBoostClassifier
fit_score("catboost", lambda: CatBoostClassifier(iterations=500, learning_rate=0.1, depth=8, l2_leaf_reg=5, thread_count=2, random_seed=SEED, verbose=0),
          lambda m: m.fit(Xf, yf), lambda m, X_: m.predict_proba(X_)[:, 1])
print(pd.DataFrame([evaluate(k, v) for k, v in HS.items()]).round(4).to_string(index=False))

hist_gbm (Phase 7): fit 60s, predict 2.3s


lightgbm: fit 76s, predict 7.7s


xgboost: fit 62s, predict 1.2s


catboost: fit 87s, predict 0.2s


             model  PR-AUC  best_thr  macro_F0.5
hist_gbm (Phase 7)  0.9972     0.700      0.9507
          lightgbm  0.9974     0.650      0.9514
           xgboost  0.9972     0.725      0.9510
          catboost  0.9969     0.625      0.9496


## 3. Deep learning — MLP with ReLU, softmax output and back-propagation
Preprocessing for networks (unlike trees): clip each feature at its 1st/99th fit percentile, standardise, replace missing values by 0 and add a **missing-value indicator** column per feature that has missing values.

In [4]:
import torch, torch.nn as nn, torch.nn.functional as Fn
torch.set_num_threads(2); torch.manual_seed(SEED)
lo, hi = np.nanpercentile(Xf, 1, axis=0), np.nanpercentile(Xf, 99, axis=0)
Xc = np.clip(Xf, lo, hi); mu, sd = np.nanmean(Xc, 0), np.nanstd(Xc, 0) + 1e-6; has_nan = np.isnan(Xf).any(0)
def prep(A):
    Z = (np.clip(A, lo, hi) - mu) / sd; M = np.isnan(Z[:, has_nan]).astype(np.float32); Z = np.nan_to_num(Z, nan=0.0)
    return np.concatenate([Z, M], 1).astype(np.float32)
Zf, Zh = prep(Xf), prep(Xh); D = Zf.shape[1]; print("network input width", D, "(", int(has_nan.sum()), "missing-indicator columns )")
class MLP(nn.Module):
    def __init__(s, d, h=(256, 128, 64), p=0.1):
        super().__init__(); L = []; k = d
        for w in h: L += [nn.Linear(k, w), nn.BatchNorm1d(w), nn.ReLU(), nn.Dropout(p)]; k = w
        s.body = nn.Sequential(*L); s.out = nn.Linear(k, 2)          # 2 classes -> softmax
    def forward(s, x): return s.out(s.body(x))
def train_mlp(epochs=8, bs=2048, lr=2e-3):
    net = MLP(D); opt = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=1e-5); sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * int(np.ceil(len(Zf) / bs)))
    Xt, yt = torch.from_numpy(Zf), torch.from_numpy(yf.astype(np.int64))
    for ep in range(epochs):
        net.train(); perm = torch.randperm(len(Xt)); tot = 0
        for i in range(0, len(Xt), bs):
            ix = perm[i:i + bs]; opt.zero_grad(); loss = Fn.cross_entropy(net(Xt[ix]), yt[ix]); loss.backward(); opt.step(); sched.step(); tot += loss.item() * len(ix)   # back-propagation
        print(f"  epoch {ep+1}/{epochs} train loss {tot/len(Xt):.4f}")
    return net
def mlp_predict(net, Z):
    net.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(Z), 65536): out.append(torch.softmax(net(torch.from_numpy(Z[i:i + 65536])), 1)[:, 1].numpy())
    return np.concatenate(out)
if "mlp_relu_softmax" not in HS:
    t0 = time.time(); net = train_mlp(); tf = time.time() - t0; t0 = time.time(); HS["mlp_relu_softmax"] = mlp_predict(net, Zh).astype(np.float64); TIMES["mlp_relu_softmax"] = {"fit_s": round(tf), "predict_s_per_M": round((time.time() - t0) / len(Zh) * 1e6, 1)}; save()
print(pd.DataFrame([evaluate(k, v) for k, v in HS.items()]).round(4).to_string(index=False))

network input width 91 ( 22 missing-indicator columns )


  epoch 1/8 train loss 0.2304


  epoch 2/8 train loss 0.0325


  epoch 3/8 train loss 0.0226


  epoch 4/8 train loss 0.0198


  epoch 5/8 train loss 0.0178


  epoch 6/8 train loss 0.0165


  epoch 7/8 train loss 0.0151


  epoch 8/8 train loss 0.0144


             model  PR-AUC  best_thr  macro_F0.5
hist_gbm (Phase 7)  0.9972     0.700      0.9507
          lightgbm  0.9974     0.650      0.9514
           xgboost  0.9972     0.725      0.9510
          catboost  0.9969     0.625      0.9496
  mlp_relu_softmax  0.9956     0.725      0.9451


## 4. Deep learning, set-aware (DeepSets over the candidates of one query)
Every candidate is encoded by the shared MLP trunk into a vector `h_i`. For each candidate the head sees `[h_i, mean_j h_j, max_j h_j]` over the *other* candidates of the same query (padding masked) and outputs a softmax over {no match, match}.
This lets the network learn "this candidate looks like the others / a better competitor exists" end to end — what Phases 6/6b tried with hand-made rules.

In [5]:
def groups(q):
    order = np.argsort(q, kind="stable"); qs = q[order]; b = np.flatnonzero(np.r_[True, qs[1:] != qs[:-1]]); e = np.r_[b[1:], len(qs)]
    return order, b, e
class SetNet(nn.Module):
    def __init__(s, d, h=(256, 128), z=64, p=0.1):
        super().__init__(); L = []; k = d
        for w in h: L += [nn.Linear(k, w), nn.LayerNorm(w), nn.ReLU(), nn.Dropout(p)]; k = w
        L += [nn.Linear(k, z), nn.ReLU()]; s.enc = nn.Sequential(*L)
        s.head = nn.Sequential(nn.Linear(3 * z, 64), nn.ReLU(), nn.Dropout(p), nn.Linear(64, 2))
    def forward(s, x, mask):                         # x: (B, L, d), mask: (B, L) bool
        h = s.enc(x); m = mask.unsqueeze(-1).float(); n = m.sum(1, keepdim=True)
        mean_all = (h * m).sum(1, keepdim=True); mean_oth = (mean_all - h * m) / (n - 1).clamp(min=1)          # mean of the OTHER candidates
        hm = h.masked_fill(~mask.unsqueeze(-1), -1e4); top2 = hm.topk(min(2, hm.shape[1]), dim=1).values
        max_oth = torch.where(h >= top2[:, :1], top2[:, -1:].expand_as(h), top2[:, :1].expand_as(h)); max_oth = torch.where(max_oth < -1e3, torch.zeros_like(max_oth), max_oth)
        return s.head(torch.cat([h, mean_oth, max_oth], -1))                                                        # (B, L, 2)
def batches(Z, yv, q, bs_q, shuffle):
    order, b, e = groups(q); idx = np.arange(len(b)); idx = np.random.permutation(idx) if shuffle else idx
    for i in range(0, len(idx), bs_q):
        sel = idx[i:i + bs_q]; Lm = int((e[sel] - b[sel]).max()); B = len(sel)
        xb = np.zeros((B, Lm, Z.shape[1]), np.float32); mb = np.zeros((B, Lm), bool); yb = np.zeros((B, Lm), np.int64); rows = np.full((B, Lm), -1)
        for j, g in enumerate(sel):
            r = order[b[g]:e[g]]; k = len(r); xb[j, :k] = Z[r]; mb[j, :k] = True; rows[j, :k] = r
            if yv is not None: yb[j, :k] = yv[r]
        yield torch.from_numpy(xb), torch.from_numpy(mb), torch.from_numpy(yb), rows
def train_set(epochs=8, bs_q=96, lr=2e-3):
    net = SetNet(D); steps = epochs * int(np.ceil(len(set(qf)) / bs_q)); opt = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=1e-5); sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps)
    for ep in range(epochs):
        net.train(); tot = n = 0
        for xb, mb, yb, _ in batches(Zf, yf, qf, bs_q, True):
            opt.zero_grad(); lo_ = net(xb, mb)[mb]; loss = Fn.cross_entropy(lo_, yb[mb]); loss.backward(); opt.step(); sched.step(); tot += loss.item() * len(lo_); n += len(lo_)
        print(f"  epoch {ep+1}/{epochs} train loss {tot/n:.4f}")
    return net
def set_predict(net, Z, q):
    net.eval(); out = np.zeros(len(Z))
    with torch.no_grad():
        for xb, mb, _, rows in batches(Z, None, q, 256, False):
            pr = torch.softmax(net(xb, mb), -1)[..., 1].numpy(); out[rows[mb.numpy()]] = pr[mb.numpy()]
    return out
if "deepsets_relu_softmax" not in HS:
    t0 = time.time(); snet = train_set(); tf = time.time() - t0; t0 = time.time(); HS["deepsets_relu_softmax"] = set_predict(snet, Zh, qh); TIMES["deepsets_relu_softmax"] = {"fit_s": round(tf), "predict_s_per_M": round((time.time() - t0) / len(Zh) * 1e6, 1)}; save()
print(pd.DataFrame([evaluate(k, v) for k, v in HS.items()]).round(4).to_string(index=False)); print(pd.DataFrame(TIMES).T)

  epoch 1/8 train loss 0.1762


  epoch 2/8 train loss 0.0280


  epoch 3/8 train loss 0.0204


  epoch 4/8 train loss 0.0172


  epoch 5/8 train loss 0.0145


  epoch 6/8 train loss 0.0130


  epoch 7/8 train loss 0.0115


  epoch 8/8 train loss 0.0108


                model  PR-AUC  best_thr  macro_F0.5
   hist_gbm (Phase 7)  0.9972     0.700      0.9507
             lightgbm  0.9974     0.650      0.9514
              xgboost  0.9972     0.725      0.9510
             catboost  0.9969     0.625      0.9496
     mlp_relu_softmax  0.9956     0.725      0.9451
deepsets_relu_softmax  0.9973     0.750      0.9491
                       fit_s  predict_s_per_M
hist_gbm (Phase 7)      60.0             12.8
lightgbm                76.0             42.1
xgboost                 62.0              6.8
catboost                87.0              0.8
mlp_relu_softmax        68.0              2.6
deepsets_relu_softmax   76.0              2.9


## 5. Blends and the decision

In [6]:
names = list(HS); ref = "hist_gbm (Phase 7)"
best_nn = max([k for k in names if k in ("mlp_relu_softmax", "deepsets_relu_softmax")], key=lambda k: average_precision_score(yh, HS[k]))
best_gb = max([k for k in names if k not in ("mlp_relu_softmax", "deepsets_relu_softmax")], key=lambda k: average_precision_score(yh, HS[k]))
HS["blend: " + ref + " + " + best_nn] = 0.5 * HS[ref] + 0.5 * HS[best_nn]
HS["blend: all boosters"] = np.mean([HS[k] for k in names if k not in ("mlp_relu_softmax", "deepsets_relu_softmax")], 0)
HS["blend: all boosters + " + best_nn] = 0.5 * np.mean([HS[k] for k in names if k not in ("mlp_relu_softmax", "deepsets_relu_softmax")], 0) + 0.5 * HS[best_nn]
R = pd.DataFrame([evaluate(k, v) for k, v in HS.items()]).set_index("model")
fe = {k: EU.macro_f05(ent_hold, ntrue, qh, yh, (HS[k] >= R.loc[k, "best_thr"]).astype(int))[1].reindex(ent_hold).values for k in HS}
rb = np.random.default_rng(SEED); idxs = [rb.integers(0, len(ent_hold), len(ent_hold)) for _ in range(1000)]
rows = []
for k in HS:
    d = fe[k] - fe[ref]; bs_ = [d[i].mean() for i in idxs]; rows.append({"model": k, "dPR-AUC": R.loc[k, "PR-AUC"] - R.loc[ref, "PR-AUC"], "dF0.5": d.mean(), "boot_lo": np.percentile(bs_, 2.5), "boot_hi": np.percentile(bs_, 97.5)})
C = pd.DataFrame(rows).set_index("model"); C["beats_reference"] = (C["dPR-AUC"] >= 0.002) & (C["dF0.5"] >= 0.003) & (C["boot_lo"] > 0)
out = R.join(C).join(pd.DataFrame(TIMES).T).round(4); print(out.to_string())
winners = C[C.beats_reference].index.tolist(); print("\nModels that clear the replacement rule:", winners if winners else "none -> keep HistGradientBoosting")

                                                   PR-AUC  best_thr  macro_F0.5  dPR-AUC   dF0.5  boot_lo  boot_hi  beats_reference  fit_s  predict_s_per_M
model                                                                                                                                                      
hist_gbm (Phase 7)                                 0.9972     0.700      0.9507   0.0000  0.0000   0.0000   0.0000            False   60.0             12.8
lightgbm                                           0.9974     0.650      0.9514   0.0001  0.0007  -0.0003   0.0018            False   76.0             42.1
xgboost                                            0.9972     0.725      0.9510   0.0000  0.0003  -0.0011   0.0016            False   62.0              6.8
catboost                                           0.9969     0.625      0.9496  -0.0004 -0.0011  -0.0030   0.0009            False   87.0              0.8
mlp_relu_softmax                                   0.9956     0.

## 6. Cascade for speed: drop obvious non-matches before computing the expensive features
Stage 1 uses only features that retrieval gives for free (`name_score, addr_score, name_rank, addr_rank, in_name, in_addr, c_is_s2, is_us`).
A candidate is skipped when stage-1 probability < τ. **τ is the largest value that keeps ≥ 99.9 % of the pairs the final model accepts (p ≥ 0.70) on the fit queries' out-of-fold hold-out**, then re-measured on the hold-out and on validation.

In [7]:
RET = ["name_score", "addr_score", "name_rank", "addr_rank", "in_name", "in_addr", "c_is_s2", "is_us"]
Rf = Fall[RET].values.astype(np.float32); Rtr, Rho = Rf[~is_hold], Rf[is_hold]
s1 = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.1, max_leaf_nodes=31, random_state=SEED).fit(Rtr, yf)
p1h = s1.predict_proba(Rho)[:, 1]; p_main = HS[ref]; acc = p_main >= 0.7
print(f"hold-out: {acc.sum():,} pairs accepted by the final-type model")
tau_grid = [1e-4, 3e-4, 1e-3, 2e-3, 5e-3, 1e-2, 2e-2]
rows = []
for tau in tau_grid:
    skip = p1h < tau; kept_acc = (acc & ~skip).sum() / acc.sum()
    f_c = EU.macro_f05(ent_hold, ntrue, qh, yh, (acc & ~skip).astype(int))[0]; rows.append({"tau": tau, "pairs skipped": skip.mean(), "accepted pairs kept": kept_acc, "macro F0.5 with cascade": f_c})
Cas = pd.DataFrame(rows); print(f"macro F0.5 without cascade: {EU.macro_f05(ent_hold, ntrue, qh, yh, acc.astype(int))[0]:.4f}"); print(Cas.round(4).to_string(index=False))
tau = float(Cas[Cas["accepted pairs kept"] >= 0.999].tau.max()) if (Cas["accepted pairs kept"] >= 0.999).any() else None; print("chosen tau (keeps >= 99.9% of accepted pairs):", tau)
if tau is not None:
    cv = pd.read_pickle(INTERIM + "/candidates_val.pkl"); sv = pd.read_pickle(INTERIM + "/val_scores.pkl"); Fv = pd.read_pickle(INTERIM + "/features_val.pkl")
    pv1 = s1.predict_proba(Fv[RET].values.astype(np.float32))[:, 1]; accv = sv.score.values >= 0.7; skipv = pv1 < tau
    tv = P4["truth"]["val"]; tv = tv[tv.s1_id.isin(set(cv.s1_id))]; ntv = tv.groupby("s1_id").size(); ev = cv.s1_id.unique().tolist()
    f0 = EU.macro_f05(ev, ntv, cv.s1_id.values, cv.label.values, accv.astype(int))[0]; f1 = EU.macro_f05(ev, ntv, cv.s1_id.values, cv.label.values, (accv & ~skipv).astype(int))[0]
    print(f"VALIDATION (tau fixed on train): pairs skipped {skipv.mean():.1%} | accepted pairs kept {(accv & ~skipv).sum()/accv.sum():.4f} | macro F0.5 {f0:.4f} -> {f1:.4f}")
    cas_res = {"tau": tau, "val_pairs_skipped": round(float(skipv.mean()), 4), "val_f05_without": round(f0, 4), "val_f05_with": round(f1, 4)}
else: cas_res = None

hold-out: 15,139 pairs accepted by the final-type model


macro F0.5 without cascade: 0.9507
   tau  pairs skipped  accepted pairs kept  macro F0.5 with cascade
0.0001         0.0000               1.0000                   0.9507
0.0003         0.0007               1.0000                   0.9507
0.0010         0.0658               0.9995                   0.9505
0.0020         0.2485               0.9971                   0.9502
0.0050         0.4812               0.9894                   0.9472
0.0100         0.6142               0.9808                   0.9448
0.0200         0.7196               0.9640                   0.9388
chosen tau (keeps >= 99.9% of accepted pairs): 0.001


VALIDATION (tau fixed on train): pairs skipped 6.6% | accepted pairs kept 0.9997 | macro F0.5 0.9508 -> 0.9507


## 7. Summary

In [8]:
summary = {"table": json.loads(out.reset_index().to_json(orient="records")), "winners": winners, "cascade": cas_res}
json.dump(summary, open("phase7b_summary.json", "w"), indent=2); print(out[["PR-AUC", "macro_F0.5", "dF0.5", "beats_reference", "predict_s_per_M"]].to_string()); print("cascade:", cas_res)

                                                   PR-AUC  macro_F0.5   dF0.5  beats_reference  predict_s_per_M
model                                                                                                          
hist_gbm (Phase 7)                                 0.9972      0.9507  0.0000            False             12.8
lightgbm                                           0.9974      0.9514  0.0007            False             42.1
xgboost                                            0.9972      0.9510  0.0003            False              6.8
catboost                                           0.9969      0.9496 -0.0011            False              0.8
mlp_relu_softmax                                   0.9956      0.9451 -0.0056            False              2.6
deepsets_relu_softmax                              0.9973      0.9491 -0.0016            False              2.9
blend: hist_gbm (Phase 7) + deepsets_relu_softmax  0.9980      0.9532  0.0025            False          